# Day 32：高级 RAG Part 1（Query Rewriting、Multi-Query、HyDE、Context Compression）

> **🎙️ 讲师旁白**：欢迎来到 Day 32 实操课堂！今天我们将通过可运行的实验，依次验证 **查询改写（`QueryRewriter`）**、**多路查询去重（`MultiQueryRetriever`）**、**假设文档检索（`HyDERetriever`）** 与 **上下文压缩（`ContextCompressor`）** 四大高级 RAG 技巧。

## 1. 环境初始化与构建企业知识库
定位仓库根目录并导入 `project_iterations/day_32` 中的 `zhishu_agent` 模块，写入包含人事、财务、运维的三条测试文档。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / 'project_iterations').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

day32_dir = repo_root / 'project_iterations' / 'day_32'
if str(day32_dir) not in sys.path:
    sys.path.insert(0, str(day32_dir))

from zhishu_agent import (
    ContextCompressor,
    Document,
    HyDERetriever,
    LocalVectorStore,
    MultiQueryRetriever,
    QueryRewriter,
)

store = LocalVectorStore(collection_name='zhishu_day32_nb')
store.add_documents([
    Document(
        id='hr-001',
        page_content=(
            '星辰数科考勤总则：全体员工遵守弹性打卡。'
            '入职满1年的员工每年享有5天带薪年假，满10年享有10天带薪年假。'
            '公司食堂每周五提供免费下午茶。'
        ),
        metadata={'department': 'hr', 'year': 2025},
    ),
    Document(
        id='fin-001',
        page_content=(
            '差旅费用报销标准：出差前须提交审批单。'
            '一线城市酒店住宿报销上限为每晚600元，需上传合规电子发票。'
        ),
        metadata={'department': 'finance', 'year': 2025},
    ),
    Document(
        id='it-001',
        page_content=(
            'Kubernetes Pod 内存溢出（OOMKilled）排查手册：退出码 137 表示触发内核 OOM。'
            '请检查容器 limits.memory 与 JVM -Xmx 参数配置，并分析 heap dump 排查内存泄漏。'
        ),
        metadata={'department': 'it', 'year': 2026},
    ),
])
print('知识库文档总数:', store.count())

知识库文档总数: 3


## 2. Query Rewriting（口语清洗与多轮指代消解）
模拟多轮对话中用户使用代词“它”追问，观察 `QueryRewriter` 如何结合 `chat_history` 改写为独立检索句。

In [2]:
rewriter = QueryRewriter()
history = [{'role': 'user', 'content': '年假制度'}]
raw_q = '请问一下它一般有几天呀？'
rewritten_q = rewriter.rewrite(raw_q, chat_history=history)
print('原始提问:', raw_q)
print('改写结果:', rewritten_q)
assert '年假制度' in rewritten_q

原始提问: 请问一下它一般有几天呀？
改写结果: 年假制度 一般有几天 带薪年假 休假天数


## 3. Multi-Query Retrieval（多视角变体生成与 RRF 去重）
单条提问往往受限于用词习惯，`MultiQueryRetriever` 生成多个视角变体并利用 RRF 融合去重。

In [3]:
base_retriever = store.as_retriever(search_kwargs={'k': 2})
mq = MultiQueryRetriever(retriever=base_retriever, num_queries=3)
mq_docs, variants = mq.retrieve_with_details('年假几天', k=2)
print('生成的查询变体:', variants)
print('RRF 去重召回 ID:', [d.id for d in mq_docs])
assert len(mq_docs) == len({d.id for d in mq_docs})

生成的查询变体: ['年假几天', '年假几天 带薪年假 休假天数', '年假 假几 几天 规范 制度 手册']
RRF 去重召回 ID: ['hr-001', 'it-001']


## 4. HyDE（假设文档向量检索）与 Context Compression（句子级上下文压缩）
先用 `HyDERetriever` 弥合短问句与长文档的语义鸿沟，再用 `ContextCompressor` 剔除切片中的无关噪音句。

In [4]:
hyde = HyDERetriever(vector_store=store, k=2)
hypo_doc, hyde_docs = hyde.retrieve_with_hypothesis('Pod 老是 OOM 怎么办')
print('生成的假设文档:', hypo_doc)
print('HyDE Top-1 召回:', hyde_docs[0].id)

compressor = ContextCompressor(min_score=0.20, max_sentences_per_doc=1)
compressed_docs = compressor.compress_documents('带薪年假有几天', mq_docs, drop_empty=True)
for d in compressed_docs:
    print(f"[{d.id}] {d.metadata['original_chars']}字 -> {d.metadata['compressed_chars']}字 | {d.page_content}")

生成的假设文档: 针对「Pod 老是 OOM 怎么办」的技术排查文档：当容器或服务发生内存溢出（OOMKilled）时，首先检查内存限制与 JVM 堆参数设置，排查内存泄漏与大对象缓存，并通过监控日志定位异常进程。
HyDE Top-1 召回: it-001
[hr-001] 68字 -> 33字 | 入职满1年的员工每年享有5天带薪年假，满10年享有10天带薪年假。
